# Part 1: Data Cleaning - Real Estate Price Analysis (City-wise)
This notebook loads the raw real estate transaction records, performs data auditing, applies standardizations, removes outliers, and saves the cleaned dataset for SQL loading and EDA.

## 1. Import Libraries and Load Raw Dataset

In [1]:
import pandas as pd
import numpy as np

# Load raw CSV dataset
raw_df = pd.read_csv('../data/raw_data.csv')
print('Raw dataset loaded successfully.')
print(f'Initial shape: {raw_df.shape}')

Raw dataset loaded successfully.
Initial shape: (76038, 9)


## 2. Check Missing Values, Duplicates, and Data Types

In [2]:
# Check missing values
print('--- Missing Values ---')
print(raw_df.isnull().sum())

# Check duplicates
duplicates_count = raw_df.duplicated().sum()
print(f'\nNumber of duplicate rows: {duplicates_count}')

# Check data types
print('\n--- Data Types ---')
print(raw_df.dtypes)

--- Missing Values ---
bhk           0
type          0
locality      0
area          0
price         0
price_unit    0
region        0
status        0
age           0
dtype: int64

Number of duplicate rows: 20312

--- Data Types ---
bhk             int64
type              str
locality          str
area            int64
price         float64
price_unit        str
region            str
status            str
age               str
dtype: object


## 3. Drop Duplicates and Handle Missing Values

In [3]:
# Drop duplicate rows
df = raw_df.drop_duplicates()
print(f'Shape after dropping duplicates: {df.shape}')

# Drop rows missing critical fields (price, area, locality, type, region)
critical_cols = ['price', 'area', 'locality', 'type', 'region']
df = df.dropna(subset=critical_cols)
print(f'Shape after dropping critical missing values: {df.shape}')

Shape after dropping duplicates: (55726, 9)
Shape after dropping critical missing values: (55726, 9)


## 4. Standardize Text Columns (Trim & Proper Case)

In [4]:
# Standardize text columns (type, locality, region, status, age)
text_cols = ['type', 'locality', 'region', 'status', 'age']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

print('Sample standardized text fields:')
print(df[text_cols].head())

Sample standardized text fields:
        type                              locality          region  \
0  Apartment   Lak And Hanware The Residency Tower    Andheri West   
1  Apartment     Radheya Sai Enclave Building No 2    Naigaon East   
2  Apartment                         Romell Serene   Borivali West   
3  Apartment  Soundlines Codename Urban Rainforest          Panvel   
4  Apartment                         Origin Oriana  Mira Road East   

               status  age  
0       Ready To Move  New  
1  Under Construction  New  
2  Under Construction  New  
3  Under Construction  New  
4  Under Construction  New  


## 5. Convert Price to `price_in_lakhs`

In [5]:
# Convert price into a single unit called price_in_lakhs (1 Cr = 100 L)
def convert_price_to_lakhs(row):
    val = row['price']
    unit = str(row['price_unit']).strip()
    if unit == 'Cr':
        return val * 100
    elif unit == 'L':
        return val
    else:
        return val

df['price_in_lakhs'] = df.apply(convert_price_to_lakhs, axis=1)
print(df[['price', 'price_unit', 'price_in_lakhs']].head())

   price price_unit  price_in_lakhs
0   2.50         Cr          250.00
1  52.51          L           52.51
2   1.73         Cr          173.00
3  59.98          L           59.98
4  94.11          L           94.11


## 6. Calculate Derived Field: `price_per_sqft`

In [6]:
# Derived column: price_per_sqft = (price_in_lakhs * 100,000) / area
df['price_per_sqft'] = (df['price_in_lakhs'] * 100000) / df['area']
print(df[['area', 'price_in_lakhs', 'price_per_sqft']].head())

   area  price_in_lakhs  price_per_sqft
0   685          250.00    36496.350365
1   640           52.51     8204.687500
2   610          173.00    28360.655738
3   876           59.98     6847.031963
4   659           94.11    14280.728376


## 7. Outlier Removal for Area and Price per Sqft (IQR Method)

In [7]:
# Outlier removal using IQR method
def remove_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return dataframe[(dataframe[column] >= lower_bound) & (dataframe[column] <= upper_bound)], lower_bound, upper_bound

# Remove outliers from area
df, lower_a, upper_a = remove_outliers_iqr(df, 'area')
print(f'Area range after IQR: [{lower_a:.2f}, {upper_a:.2f}]')

# Remove outliers from price_per_sqft
df, lower_p, upper_p = remove_outliers_iqr(df, 'price_per_sqft')
print(f'Price per sqft range after IQR: [{lower_p:.2f}, {upper_p:.2f}]')
print(f'Shape after outlier removal: {df.shape}')

Area range after IQR: [-112.50, 1907.50]
Price per sqft range after IQR: [-8219.96, 35279.96]
Shape after outlier removal: (51358, 11)


## 8. Filter BHK values (Keep only 1 to 10)

In [8]:
# Remove unrealistic BHK values (keep 1 to 10)
initial_count = len(df)
df = df[(df['bhk'] >= 1) & (df['bhk'] <= 10)]
final_count = len(df)
print(f'Rows before BHK filter: {initial_count}')
print(f'Rows after BHK filter: {final_count}')
print(f'Removed {initial_count - final_count} records with BHK out of range [1, 10].')

Rows before BHK filter: 51358
Rows after BHK filter: 51358
Removed 0 records with BHK out of range [1, 10].


## 9. Final Shape and Summary Statistics

In [9]:
print(f'Final Cleaned Dataset Shape: {df.shape}')
print('\n--- Summary Statistics ---')
print(df.describe())

Final Cleaned Dataset Shape: (51358, 11)

--- Summary Statistics ---
                bhk          area         price  price_in_lakhs  \
count  51358.000000  51358.000000  51358.000000    51358.000000   
mean       1.813875    878.404494     33.176428      121.211732   
std        0.744365    337.231318     33.700989       87.838663   
min        1.000000    136.000000      1.000000        4.490000   
25%        1.000000    630.000000      1.740000       59.952500   
50%        2.000000    806.000000     24.800000       95.000000   
75%        2.000000   1080.000000     63.000000      160.000000   
max        5.000000   1907.000000     99.990000      650.000000   

       price_per_sqft  
count    51358.000000  
mean     13480.816010  
std       7105.590245  
min        646.766169  
25%       8000.000000  
50%      11614.143921  
75%      18275.862069  
max      35268.505080  


## 10. Export Cleaned Dataset to CSV

In [10]:
output_path = '../data/cleaned_data.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned dataset exported to {output_path} successfully!')

Cleaned dataset exported to ../data/cleaned_data.csv successfully!
